#Data Generation*

In [36]:
import random

def generate_dataset(n, N=10000, low=0, high=20, filename="data.txt"):
    with open(filename, "w") as f:
        for _ in range(N):
            rect = []
            for _ in range(n):
                a = random.randint(low, high)
                b = random.randint(a, high)
                rect.extend([a, b])
            f.write(" ".join(map(str, rect)) + "\n")
generate_dataset(n=2, filename="data_n2.txt")

#R-Tree with Quadratic Split

In [37]:
class Rectangle:
    def __init__(self, mins, maxs):
        self.mins = mins
        self.maxs = maxs

class Node:
    def __init__(self, leaf=True):
        self.leaf = leaf
        self.entries = []
        self.children = []
        self.mbr = None

In [38]:
#BUILD & STORE R-TREES FOR ALL n
trees_by_n = {}

dims = [2, 4, 8, 16, 32]

for n in dims:
    tree = RTree(M=10, m=4)

    data_file = f"data_n{n}.txt"

    with open(data_file, "r") as f:
        for line in f:
            vals = list(map(int, line.split()))
            mins = vals[::2]
            maxs = vals[1::2]
            rect = Rectangle(mins, maxs)
            tree.insert(rect)

    trees_by_n[n] = tree

In [39]:
def combine_mbr(r1, r2): #MBR Utilities
    mins = [min(a, b) for a, b in zip(r1.mins, r2.mins)]
    maxs = [max(a, b) for a, b in zip(r1.maxs, r2.maxs)]
    return Rectangle(mins, maxs)

def area(rect):
    vol = 1
    for mn, mx in zip(rect.mins, rect.maxs):
        vol *= (mx - mn)
    return vol

In [40]:
def quadratic_split(entries, M, m): #Quadratic Split
    max_waste = -1
    seed1, seed2 = None, None

    for i in range(len(entries)):
        for j in range(i+1, len(entries)):
            r1, r2 = entries[i], entries[j]
            combined = combine_mbr(r1, r2)
            waste = area(combined) - area(r1) - area(r2)
            if waste > max_waste:
                max_waste = waste
                seed1, seed2 = r1, r2

    group1 = [seed1]
    group2 = [seed2]
    remaining = [e for e in entries if e not in (seed1, seed2)]

    while remaining:
        e = remaining.pop()
        inc1 = area(combine_mbr(group1[0], e)) - area(group1[0])
        inc2 = area(combine_mbr(group2[0], e)) - area(group2[0])
        if inc1 < inc2:
            group1.append(e)
        else:
            group2.append(e)

    return group1, group2

In [41]:
class RTree: #insert logic
    def __init__(self, M=10, m=4):
        self.root = Node()
        self.M = M
        self.m = m

    def insert(self, rect):
        self.root.entries.append(rect)
        if len(self.root.entries) > self.M:
            g1, g2 = quadratic_split(self.root.entries, self.M, self.m)
            self.root.entries = g1
            new_node = Node()
            new_node.entries = g2

#Nearest Neighbor Search

In [42]:
import math
#distance to recatngle
def dist_point_rect(point, rect):
    d = 0
    for p, mn, mx in zip(point, rect.mins, rect.maxs):
        if p < mn: d += (mn - p) ** 2
        elif p > mx: d += (p - mx) ** 2
    return math.sqrt(d)

In [43]:
import time
#nn search
def nn_search(tree, query):
    start = time.time()
    best = float("inf")
    visited = 0

    for rect in tree.root.entries:
        visited += 1
        d = dist_point_rect(query, rect)
        best = min(best, d)

    return best, visited, time.time() - start

#Result

In [44]:
import pandas as pd
results = []
for n in [2, 4, 8, 16, 32]:
    generate_dataset(n, filename=f"data_n{n}.txt")
    tree = RTree()

    with open(f"data_n{n}.txt") as f:
        for line in f:
            vals = list(map(int, line.split()))
            mins = vals[::2]
            maxs = vals[1::2]
            tree.insert(Rectangle(mins, maxs))

    query = [random.randint(0,20) for _ in range(n)]
    dist, visited, t = nn_search(tree, query)

    results.append([n, t, visited])

df = pd.DataFrame(results, columns=["Dimensions", "Time", "NodesVisited"])
df

,Dimensions,Time,NodesVisited
0,2,0.000021,3
1,4,0.000017,4
2,8,0.000019,6
3,16,0.000030,8
4,32,0.000028,5


In [46]:
import pandas as pd

#df has columns: Dimensions, Time, NodesVisited
#Time is in seconds (as shown 0.000021), convert to milliseconds

final_df = df.copy()

final_df.insert(0, "Srl. No.", range(1, len(final_df) + 1))
final_df["Average Time taken (msec) (T)"] = (final_df["Time"] * 1000.0).round(3)
final_df["Average no. of Nodes visited (V)"] = final_df["NodesVisited"].round(2)

final_df = final_df.rename(columns={"Dimensions": "Number of Dimensions (n)"})

final_df = final_df[[
    "Srl. No.",
    "Number of Dimensions (n)",
    "Average Time taken (msec) (T)",
    "Average no. of Nodes visited (V)"
]]

display(final_df)
final_df.to_csv("RTree_Assignment_Table.csv", index=False)


,Srl. No.,Number of Dimensions (n),Average Time taken (msec) (T),Average no. of Nodes visited (V)
0,1,2,0.021,3
1,2,4,0.017,4
2,3,8,0.019,6
3,4,16,0.030,8
4,5,32,0.028,5


In [47]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

#final_df must already exist (your formatted table dataframe)

cols = list(final_df.columns)
data = final_df.values.tolist()

fig, ax = plt.subplots(figsize=(11.7, 4.5))  #A4 landscape-ish
ax.axis("off")

tbl = ax.table(
    cellText=data,
    colLabels=cols,
    cellLoc="center",
    loc="center"
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 1.8)

with PdfPages("RTree_Final_Result_Table.pdf") as pdf:
    pdf.savefig(fig, bbox_inches="tight")
plt.close(fig)

In [48]:
import os
print(os.path.abspath("RTree_Final_Result_Table.pdf"))
print("Exists:", os.path.exists("RTree_Final_Result_Table.pdf"))
print("Size (bytes):", os.path.getsize("RTree_Final_Result_Table.pdf"))

/content/RTree_Final_Result_Table.pdf
Exists: True
Size (bytes): 14745


In [49]:
from google.colab import files
files.download("RTree_Final_Result_Table.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>